In [ ]:
import pandas as pd
import ast
from collections import defaultdict
import re

# 1. Load data
df = pd.read_csv("alias_jw_85.csv")

def parse_aliases(val):
    try:
        return ast.literal_eval(val)
    except:
        return [str(val).strip()]

df["aliases"] = df["AliasCharacters"].apply(parse_aliases)

ROLE_KEYWORDS = [
    # ===== KELUARGA INTI =====
    ('meme', 'ibu'),
    ('i meme', 'ibu'),
    ('memen', 'ibu'),
    ('biang', 'ibu'),
    ('biange', 'ibu'),
    ('i biang', 'ibu'),

    ('bapa', 'ayah'),
    ('bapan', 'ayah'),
    ('bapa i', 'ayah'),
    ('aji', 'ayah'),

    # ===== KAKEK / NENEK =====
    ('dadong', 'nenek'),
    ('i dadong', 'nenek'),
    ('dadong jepun', 'nenek'),
    ('dadong getget', 'nenek'),

    ('pekak', 'kakek'),
    ('pekak dukuh', 'kakek'),
    ('i pekak dukuh', 'kakek'),
    ('kaki', 'kakek'),
    ('i kaki', 'kakek'),
    ('i kaki buyut', 'kakek'),
    ('i kaki dukuh', 'kakek'),
    ('i kaki perodong', 'kakek'),

    # ===== ANAK =====
    ('pianak', 'anak'),
    ('pianake', 'anak'),
    ('pianaknya', 'anak'),
    ('pianakne', 'anak'),
    ('pianak ipun', 'anak'),
    ('pianak ipu', 'anak'),
    ('pianak titiange', 'anak'),
    ('pianak muani', 'anak laki-laki'),
    ('pianak-pianak', 'anak-anak'),

    ('panak', 'anak'),
    ('panak icange', 'anak'),
    ('panak jara', 'anak'),

    ('rare', 'anak'),
    ('rare punika', 'anak'),
    ('i rare angon', 'anak'),

    ('cening', 'anak'),
    ('i cening', 'anak'),
    ('ceninge', 'anak'),
    ('cening durma', 'anak'),

    # ===== SAUDARA =====
    ('beli', 'kakak'),
    ('bli', 'kakak'),
    ('beli wayan', 'kakak'),
    ('beli empas', 'kakak'),
    ('bli menceng', 'kakak'),

    ('adi', 'adik'),
    ('adin', 'adik'),
    ('adi bintang lara', 'adik'),

    ('rakan ceninge', 'saudara'),

    # ===== PEREMPUAN / ANAK PEREMPUAN =====
    ('luh', 'anak perempuan'),
    ('anak luh', 'anak perempuan'),
    ('anake luh', 'anak perempuan'),
    ('anake luh ento', 'anak perempuan'),
    ('ni luh', 'anak perempuan'),
    ('ni luh arum', 'anak perempuan'),
    ('ni luh lancah', 'anak perempuan'),
    ('luh arum', 'anak perempuan'),
    ('luh sari', 'anak perempuan'),
    ('luh lancah', 'anak perempuan'),
    ('luh ayu kantrungan', 'anak perempuan'),

    # ===== BANGSAWAN / KERAJAAN =====
    ('ida sang prabu', 'raja'),
    ('sang prabu', 'raja'),
    ('ratu sang prabu', 'raja'),
    ('gusti prabu', 'raja'),
    ('prabu singa', 'raja'),
    ('sang prabu singa', 'raja'),
    ('ida sang prabu singa', 'raja'),
    ('ida sang prabu daha', 'raja'),
    ('ida sang prabu mangsalia', 'raja'),
    ('prabu suliawana', 'raja'),
    ('sang prabu maya danawa', 'raja'),

    ('ratu', 'raja/bangsawan'),
    ('i ratu', 'raja/bangsawan'),
    ('para ratu', 'raja/bangsawan'),
    ('ratu anak agung', 'bangsawan'),
    ('ratu dewa agung', 'bangsawan'),
    ('ratu dewa ayu', 'bangsawan'),
    ('ratun katak', 'bangsawan'),
    ('i ratun katak', 'bangsawan'),
    ('ratun katake', 'bangsawan'),

    ('anake agung', 'bangsawan'),
    ('anak agung', 'bangsawan'),
    ('ida anake agung', 'bangsawan'),
    ('ida anak agung ngurah rangsasa', 'bangsawan'),
    ('okan anak agung', 'bangsawan'),
    ('agung', 'bangsawan'),

    ('raden galuh', 'putri bangsawan'),
    ('ida raden galuh', 'putri bangsawan'),
    ('i raden galuh', 'putri bangsawan'),
    ('raden galuh pitu', 'putri bangsawan'),
    ('raden galuhe', 'putri bangsawan'),
    ('ida raden galuh daha', 'putri bangsawan'),
    ('galuh daha', 'putri bangsawan'),
    ('i galuh daha', 'putri bangsawan'),
    ('i galuh', 'putri bangsawan'),
    ('galuh pitu', 'putri bangsawan'),
    ('galuh sumanasa', 'putri bangsawan'),
    ('biang galuh', 'ibu bangsawan'),
    ('biang raden galuh', 'ibu bangsawan'),

    ('raden mantri', 'pangeran/menteri'),
    ('ida raden mantri', 'pangeran/menteri'),
    ('i raden mantri', 'pangeran/menteri'),
    ('i mantri', 'menteri'),
    ('mantri koripan', 'menteri'),
    ('i mantri koripan', 'menteri'),
    ('tandamantri', 'menteri'),
    ('ida raden jajarpikatan', 'bangsawan'),
    ('raden jajarpikatan', 'bangsawan'),

    ('prameswari', 'permaisuri'),

    # ===== KEAGAMAAN / KETUHANAN =====
    ('ida bhatara', 'dewa'),
    ('ida bhatara indra', 'dewa'),
    ('ida bhatara brahma', 'dewa'),
    ('bhatara wisnu', 'dewa'),
    ('ratu betara', 'dewa'),
    ('ratu bhatara', 'dewa'),

    # ===== JABATAN ADAT / DESA =====
    ('jero', 'orang terpandang/adat'),
    ('jro', 'orang terpandang/adat'),
    ('jero kelian', 'pemimpin adat'),
    ('jro kelian', 'pemimpin adat'),
    ('jero klian', 'pemimpin adat'),
    ('jro kelihan desa', 'pemimpin adat'),
    ('jro kelian banjar', 'pemimpin adat'),
    ('kelian banjar', 'pemimpin adat'),
    ('jero bendesa', 'pemimpin adat'),
    ('jero desa', 'tokoh desa'),
    ('panyarikan desa', 'sekretaris desa'),
    ('jero pecalang', 'pecalang'),
    ('jero mangku', 'tokoh agama'),
    ('jero dukun', 'dukun'),
    ('jero anak lingsir', 'tetua'),

    ('krama', 'warga'),
    ('krama desa', 'warga desa'),
    ('krama banjar', 'warga banjar'),
    ('krama banjare', 'warga banjar'),
    ('desa', 'warga desa'),

    # ===== PROFESI / PERAN SOSIAL =====
    ('pande', 'pandai besi/tukang'),
    ('pareka', 'abdi/pelayan'),
    ('parekan', 'abdi/pelayan'),
    ('i parekan', 'abdi/pelayan'),
    ('parekanida', 'abdi/pelayan'),
    ('makakalih pareka', 'abdi/pelayan'),

    ('panjak', 'abdi'),
    ('panjake', 'abdi'),
    ('panjaknya', 'abdi'),
    ('para panjake', 'abdi'),
    ('panjak-panjak ida', 'abdi'),
    ('panjak ida bhatara indra', 'abdi'),
    ('panjak-panjak puri kertha bumi', 'abdi'),

    # ===== SAPAAN KHUSUS / LAINNYA =====
    ('me', 'ibu/perempuan dewasa'),
    ('me lutung', 'ibu/perempuan dewasa'),
    ('meme lutung', 'ibu'),
    ('men sugih', 'ibu/perempuan dewasa'),
    ('men tiwas', 'ibu/perempuan dewasa'),
    ('men bekung', 'ibu/perempuan dewasa'),
    ('men bekunge', 'ibu/perempuan dewasa'),
    ('men poleng', 'ibu/perempuan dewasa'),
    ('men jempenit', 'ibu/perempuan dewasa'),
    ('men cubling', 'ibu/perempuan dewasa'),
    ('men brayut', 'ibu/perempuan dewasa'),

    ('pan', 'ayah/laki-laki dewasa'),
    ('pan karsa', 'ayah/laki-laki dewasa'),
    ('pan cening', 'ayah/laki-laki dewasa'),
    ('pan sari', 'ayah/laki-laki dewasa'),
    ('pan meri', 'ayah/laki-laki dewasa'),
    ('meri pan meri', 'ayah/laki-laki dewasa'),
    ('pan mangut', 'ayah/laki-laki dewasa'),
    ('pan jempenit', 'ayah/laki-laki dewasa'),
    ('pan brayut', 'ayah/laki-laki dewasa'),
    ('pan bergah', 'ayah/laki-laki dewasa'),
    ('pan badung', 'ayah/laki-laki dewasa'),
    ('pan bekung', 'ayah/laki-laki dewasa'),
    ('pan poleng', 'ayah/laki-laki dewasa'),
    ('pan balang tamak', 'ayah/laki-laki dewasa'),
    ('pan balang tamake', 'ayah/laki-laki dewasa'),
    ('pan angklung gadang', 'ayah/laki-laki dewasa'),
    ('pan angklung gadange', 'ayah/laki-laki dewasa'),
    ('bapa angklung gadang', 'ayah'),
]

ORDINAL_PATTERN = re.compile(
    r'\bke(?:-?\d+|satu|dua|tiga|empat|lima|enam|tujuh|delapan|sembilan|sepuluh|belas|puluh|ratus|ribu)\b',
    re.IGNORECASE
)

def normalize(text):
    return text.lower().strip()

def get_role(alias):
    alias = normalize(alias)
    words = alias.split()
    for keyword, role in ROLE_KEYWORDS:
        if alias == keyword or keyword in words:
            return role
    return None

def contains_ordinal(alias):
    return bool(ORDINAL_PATTERN.search(normalize(alias)))

# 4. Pengelompokan (Clustering)
grouped = defaultdict(list)
for _, row in df.iterrows():
    grouped[row['StoryID']].append({
        'person': row['CharactersID'],
        'aliases': [normalize(a) for a in row['aliases']]
    })

final_results = []
for story_id, clusters in grouped.items():
    merged = []
    visited = [False] * len(clusters)

    for i in range(len(clusters)):
        if visited[i]: continue

        current_aliases = set(clusters[i]['aliases'])
        merged_indices = [i]
        roles_i = {get_role(a) for a in current_aliases if get_role(a)}

        for j in range(i + 1, len(clusters)):
            if visited[j]: continue

            other_aliases = set(clusters[j]['aliases'])
            roles_j = {get_role(a) for a in other_aliases if get_role(a)}

            # Jangan gabungkan jika ada penanda urutan (ordinal)
            if any(contains_ordinal(a) for a in current_aliases | other_aliases):
                continue

            # Gabungkan jika peran sama
            if roles_i and roles_i == roles_j:
                current_aliases |= other_aliases
                visited[j] = True
                merged_indices.append(j)

        merged.append({
            "aliases": list(current_aliases),
            "role": next(iter(roles_i)) if roles_i else None
        })
        for idx in merged_indices:
            visited[idx] = True

    for idx, group in enumerate(merged, 1):
        final_results.append({
            "storyID": story_id,
            "characterID": f"Tokoh-{idx}",
            "AliasCharacters": sorted(set(group['aliases']))
        })

# 5. Simpan Hasil
df_out = pd.DataFrame(final_results)
df_out.to_csv("alias_jw_85_wsm.csv", index=False)
print("✅ Selesai! File disimpan sebagai alias_jw_85_wsm.csv")


In [ ]:
import pandas as pd
import ast
import re
from itertools import combinations
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# =========================
# KONFIGURASI FILE
# =========================
GROUND_TRUTH_CSV = "DATA SATYA/string_similarity_bali.csv"  # GT
PREDICT_CSV      = "alias_jw_85_wsm.csv"           # Prediksi

# Nama kolom standar (otomatis dinormalkan kalau beda casing)
COL_STORY   = "StoryID"
COL_CLUSTER = "CharactersID"
COL_ALIASES = "AliasCharacters"

# =========================
# PARSER & NORMALIZER
# =========================
_punct_tail = re.compile(r"[,\.;:!\?\)]*$")  # hapus tanda baca di ekor token

def norm_alias(s: str) -> str:
    return _punct_tail.sub("", str(s).strip().lower())

def parse_aliases_cell(x):
    """Dukung:
       - list Python (['a','b'])
       - stringified list ("['a','b']")
       - string dipisah koma/semicolon ("a, b; c")
       - string tunggal ("a")"""
    if isinstance(x, list):
        return [norm_alias(t) for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                lst = ast.literal_eval(s)
                return [norm_alias(t) for t in lst if str(t).strip()]
            except Exception:
                pass
        # split by comma/semicolon jika ada
        if ("," in s) or (";" in s):
            parts = re.split(r"[;,]", s)
            return [norm_alias(t) for t in parts if str(t).strip()]
        # satu alias
        return [norm_alias(s)] if s else []
    return []

def read_clusters(csv_path):
    """Kembalikan dict: {StoryID: {ClusterID: [alias,…]}}"""
    df = pd.read_csv(csv_path)

    # selaraskan nama kolom jika beda penulisan
    rename_map = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "storyid":      rename_map[c] = COL_STORY
        if lc in ("charactersid","person","cluster","tokoh"): rename_map[c] = COL_CLUSTER
        if lc in ("aliascharacters","aliases","alias","listalias","characters"): rename_map[c] = COL_ALIASES
    if rename_map:
        df = df.rename(columns=rename_map)

    for must in [COL_STORY, COL_CLUSTER, COL_ALIASES]:
        if must not in df.columns:
            raise KeyError(f"Kolom wajib '{must}' tidak ada di {csv_path}. Kolom: {list(df.columns)}")

    df[COL_ALIASES] = df[COL_ALIASES].apply(parse_aliases_cell)

    clusters_per_story = {}
    for sid, sub in df.groupby(COL_STORY):
        story = {}
        for _, r in sub.iterrows():
            cid = str(r[COL_CLUSTER])
            aliases = r[COL_ALIASES]
            if not aliases: 
                continue
            story.setdefault(cid, set()).update(aliases)
        # set → list & sort
        story = {k: sorted(v) for k, v in story.items() if v}
        if story:
            clusters_per_story[sid] = story
    return clusters_per_story

# =========================
# BACA DATA → gt, pr
# =========================
gt = read_clusters(GROUND_TRUTH_CSV)
pr = read_clusters(PREDICT_CSV)

# =========================
# EVALUASI (pairwise + ARI/NMI)
# =========================
def to_label_map(cluster_dict):
    m = {}
    for i, (cid, aliases) in enumerate(sorted(cluster_dict.items(), key=lambda x: str(x[0]))):
        for a in aliases:
            m[a] = i
    return m

def pairwise_confusion(gt_map, pr_map):
    aliases = sorted(set(gt_map) & set(pr_map))
    if len(aliases) < 2:
        return 0,0,0,0,0
    TP=FP=FN=TN=0; total=0
    for i in range(len(aliases)):
        for j in range(i+1, len(aliases)):
            a, b = aliases[i], aliases[j]
            sg = (gt_map[a]==gt_map[b])
            sp = (pr_map[a]==pr_map[b])
            total += 1
            if  sg and  sp: TP += 1
            elif not sg and sp: FP += 1
            elif sg and not sp: FN += 1
            else: TN += 1
    return TP,FP,FN,TN,total

def safe_div(a,b): return a/b if b else 0.0
def metrics(TP,FP,FN,TN):
    precision = safe_div(TP, TP+FP)
    recall    = safe_div(TP, TP+FN)
    f1        = safe_div(2*precision*recall, precision+recall)
    accuracy  = safe_div(TP+TN, TP+FP+FN+TN)
    return accuracy, precision, recall, f1

# Per-story, macro, micro, ARI/NMI
rows=[]
micro_TP=micro_FP=micro_FN=micro_TN=0
y_true=[]; y_pred=[]

common_sids = sorted(set(gt) & set(pr))
if not common_sids:
    raise ValueError("Tidak ada StoryID yang sama di GT dan Prediksi.")

for sid in common_sids:
    gt_map = to_label_map(gt[sid])
    pr_map = to_label_map(pr[sid])

    TP,FP,FN,TN,total = pairwise_confusion(gt_map, pr_map)
    if total==0:
        continue

    acc, prec, rec, f1 = metrics(TP,FP,FN,TN)
    rows.append({
        "StoryID": sid,
        "pairs": total,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

    micro_TP += TP; micro_FP += FP; micro_FN += FN; micro_TN += TN

    common_aliases = sorted(set(gt_map) & set(pr_map))
    y_true.extend([gt_map[a] for a in common_aliases])
    y_pred.extend([pr_map[a] for a in common_aliases])

# Per-story report
df_report = pd.DataFrame(rows).sort_values(["StoryID","pairs"], ascending=[True,False]).reset_index(drop=True)
print("=== Per-Story Report (head) ===")
print(df_report.head(10))

# Macro average
macro_acc  = df_report["Accuracy"].mean()  if not df_report.empty else 0.0
macro_prec = df_report["Precision"].mean() if not df_report.empty else 0.0
macro_rec  = df_report["Recall"].mean()    if not df_report.empty else 0.0
macro_f1   = df_report["F1"].mean()        if not df_report.empty else 0.0

# Micro average (overall)
overall_acc, overall_prec, overall_rec, overall_f1 = metrics(micro_TP, micro_FP, micro_FN, micro_TN)

# ARI & NMI
overall_ari = adjusted_rand_score(y_true, y_pred) if y_true else 0.0
overall_nmi = normalized_mutual_info_score(y_true, y_pred) if y_true else 0.0

print("\n=== Macro Average (per-story mean) ===")
print(f"Accuracy : {macro_acc:.4f}")
print(f"Precision: {macro_prec:.4f}")
print(f"Recall   : {macro_rec:.4f}")
print(f"F1       : {macro_f1:.4f}")

print("\n=== Overall (Micro) ===")
print(f"Accuracy : {overall_acc:.4f}")
print(f"Precision: {overall_prec:.4f}")
print(f"Recall   : {overall_rec:.4f}")
print(f"F1       : {overall_f1:.4f}")

print("\n=== Information-Theoretic ===")
print(f"ARI      : {overall_ari:.4f}")
print(f"NMI      : {overall_nmi:.4f}")